In [4]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage,SystemMessage
from langchain_core.prompts.chat import ChatPromptTemplate, MessagesPlaceholder, HumanMessagePromptTemplate, SystemMessagePromptTemplate


# 定义系统消息
system_message = SystemMessagePromptTemplate.from_template(
    """
    你是一个负责管理电网的智能代理，专门用于规划电网操作。
    你的主要任务包括创建电网环境、初始化环境图、绘制电网布局以及进行发电机功率的重新分配。
    你需要根据用户提供的信息，规划出一系列具体的操作步骤。

    可用的操作包括：
    - "create_environment": 创建一个新的 grid2op 环境，并设置所需的后端参数。
    - "initialize_graph": 在现有的环境中初始化电网的图形表示，并提取最初的观测结果。
    - "plot_grid": 绘制当前电网的布局图，并能够将图像输出以供查看。
    - "redispatch": 根据电网的实时需求和发电情况，重新分配各个发电机的输出功率。

    你需要根据用户的描述，分析并规划出完成任务所需的具体步骤，包括操作的顺序和每个步骤的详细说明,请拆分成非常细的步骤。
    """
)

# 定义用户消息
user_message = HumanMessagePromptTemplate.from_template(
    """
    根据以下指令，请你规划出详细的电网操作步骤：
    {user_input}
    请确保操作计划明确、完整，并包括所有必要的细节。
    请你使用如下JSON格式回复：
    {{
      "planned_steps": [
        {{
          "step_number": 1,
          "operation": "创建新的 grid2op 环境",
          "action_guidance": "调用 create_environment()，确保所有初始化参数已正确设置。"
        }},
        {{
          "step_number": 2,
          "operation": "初始化电网环境的图形表示",
          "action_guidance": "执行 initialize_graph()，加载当前电网配置并生成图形输出。"
        }},
        {{
          "step_number": 3,
          "operation": "绘制电网布局图",
          "action_guidance": "使用 plot_grid() 绘制和输出电网布局，确保所有元件都清晰可见。"
        }},
        {{
          "step_number": 4,
          "operation": "重新分配发电机功率",
          "action_guidance": "调用 redispatch(generator_id, power_adjustment)，根据需求和容量调整各发电机输出。"
        }}
      ]
    }}
        
    """
)



# 组合提示模板
prompt = ChatPromptTemplate.from_messages([system_message, user_message])

# 配置Languange Model
llm = ChatOpenAI(model="deepseek-chat", temperature=0, openai_api_base="https://api.deepseek.com", openai_api_key="sk-8e3a75c3f4f54d9c9fd7dd779dcd80a8")

# 绑定模型与响应格式
planning_agent = prompt | llm.bind(response_format={"type": "json_object"})

msg = planning_agent.invoke("我想对所有的发电机功率随机调配5M瓦的功率，每次操作都画图一次，一共进行五次")
print(msg)

content='{\n  "planned_steps": [\n    {\n      "step_number": 1,\n      "operation": "创建新的 grid2op 环境",\n      "action_guidance": "调用 create_environment()，确保所有初始化参数已正确设置。"\n    },\n    {\n      "step_number": 2,\n      "operation": "初始化电网环境的图形表示",\n      "action_guidance": "执行 initialize_graph()，加载当前电网配置并生成图形输出。"\n    },\n    {\n      "step_number": 3,\n      "operation": "绘制电网布局图",\n      "action_guidance": "使用 plot_grid() 绘制和输出电网布局，确保所有元件都清晰可见。"\n    },\n    {\n      "step_number": 4,\n      "operation": "重新分配发电机功率",\n      "action_guidance": "调用 redispatch(generator_id, power_adjustment)，根据需求和容量调整各发电机输出。"\n    },\n    {\n      "step_number": 5,\n      "operation": "绘制电网布局图",\n      "action_guidance": "使用 plot_grid() 绘制和输出电网布局，确保所有元件都清晰可见。"\n    },\n    {\n      "step_number": 6,\n      "operation": "重新分配发电机功率",\n      "action_guidance": "调用 redispatch(generator_id, power_adjustment)，根据需求和容量调整各发电机输出。"\n    },\n    {\n      "step_number": 7,\n      "operation": "绘制电网布局图",\n      "acti

In [16]:
from IPython.display import Markdown
import json

l = json.loads(msg.content)
type(l)
print(l["planned_steps"][0])

{'step_number': 1, 'operation': '创建新的 grid2op 环境', 'action_guidance': '调用 create_environment()，确保所有初始化参数已正确设置。'}
